# DATSCI7030 — Repository Audit & Founder Action Report

**Prepared for:** Ibrahim Haroun (Founder / Author)
**Prepared by:** Senior Data Science Architect (project assistant)
**Date:** 2026-07-04
**Scope:** Full repository scan against the project's own governing principles — reproducible, modular, well documented, version controlled, research focused, dissertation friendly.

---

### How to use this notebook

This is a living document, not a one-off memo. It is saved at the repo root so it travels with the project.

- Each issue follows the required review format: **Problem → Reason → Recommended Change → Files Affected → Implementation Steps → Dissertation Impact**.
- Add your own notes directly under any finding as a markdown blockquote starting with `> Founder note:` — easy to write, easy to grep for later, and renders cleanly.
- Checkboxes in the Action Plan (Section 9) are real markdown checkboxes — tick them as you close items out, then re-run Section 0's scan cell to confirm the underlying issue is actually resolved (don't just tick and move on).
- **Nothing in the repository has been deleted, renamed, or overwritten to produce this report.** Every recommendation below is a proposal awaiting your decision — consistent with the project rule to analyse before acting.

**Priority key:** 🔴 P0 = blocks credibility or safety, fix first · 🟠 P1 = fix before submission · 🟡 P2 = polish / optional.

**Headline finding:** the pipeline logic (event study → causal inference → feature engineering → ML → SHAP) is substantive and mostly complete through Phase 5, but the repository has three compounding problems that will cost marks independently of the modelling: **(1)** no version control at all, **(2)** live credentials sitting in plaintext files, and **(3)** no true market-only baseline exists yet to answer RQ3. Everything else is fixable in an afternoon; those three are not — they need decisions from you.


## 0. Live Repository Scan

Read-only, no side effects. Run this first, and re-run any time to check progress — execute from the repo root.

In [1]:

import os, re
from pathlib import Path
from datetime import datetime

ROOT = Path(".").resolve()
print("Scanning:", ROOT)

def section(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

# --- 1. Duplicate / near-duplicate notebooks -----------------------------
section("1. Notebook duplicates (by phase prefix)")
nb_dir = ROOT / "notebooks"
if nb_dir.exists():
    by_prefix = {}
    for f in sorted(nb_dir.glob("*.ipynb")):
        prefix = re.match(r"[_0-9]*(\d+)", f.stem)
        key = prefix.group(1) if prefix else f.stem
        by_prefix.setdefault(key, []).append(f)
    for key, files in by_prefix.items():
        if len(files) > 1:
            print(f"  Phase {key}: {len(files)} candidate files")
            for f in files:
                print(f"    - {f.name}  ({f.stat().st_size/1024:.0f} KB)")

# --- 2. Duplicate report drafts ------------------------------------------
section("2. Report / dissertation draft duplicates")
for pattern in ["reports/*.docx", "reports/dissertation/*.docx"]:
    matches = sorted(ROOT.glob(pattern))
    if matches:
        print(f"  {pattern} -> {len(matches)} file(s)")
        for f in matches:
            mtime = datetime.fromtimestamp(f.stat().st_mtime)
            print(f"    - {f.relative_to(ROOT)}  ({f.stat().st_size/1024:.0f} KB, modified {mtime:%Y-%m-%d})")

# --- 3. Missing README.md in key folders ---------------------------------
section("3. Missing README.md")
key_folders = ["notebooks", "src", "models", "scripts", "tests", "reports",
               "reports/figures", "reports/dissertation", "data", "docs"]
for folder in key_folders:
    p = ROOT / folder
    if p.exists() and not (p / "README.md").exists():
        print(f"  MISSING: {folder}/README.md")

# --- 4. Secret / credential exposure sweep -------------------------------
section("4. Potential secrets committed in plaintext")
secret_patterns = [
    re.compile(r"(api_key|apikey|token|password|secret)\s*[:=]\s*['\"]?[A-Za-z0-9_\-]{12,}", re.I),
    re.compile(r"^\s*\w+\s*:\s*['\"][A-Za-z0-9]{16,}['\"]", re.M),  # bare quoted long tokens, e.g. under an api_keys: block
]
scan_files = list(ROOT.glob("*.yaml")) + list(ROOT.glob("*.yml")) + \
             list((ROOT / "notebooks").glob(".env")) + list(ROOT.glob(".env"))
for f in scan_files:
    try:
        text = f.read_text(errors="ignore")
    except Exception:
        continue
    for pat in secret_patterns:
        for m in pat.finditer(text):
            snippet = m.group(0)
            masked = snippet[:20] + "...<redacted>"
            line_no = text[:m.start()].count("\n") + 1
            print(f"  {f.relative_to(ROOT)}:{line_no}  {masked}")

# --- 5. Version control check --------------------------------------------
section("5. Version control")
print("  .git present:", (ROOT / ".git").exists())

# --- 6. Filesystem clutter -------------------------------------------------
section("6. Clutter (.DS_Store / .ipynb_checkpoints)")
ds = list(ROOT.rglob(".DS_Store"))
ckpt = [p for p in ROOT.rglob(".ipynb_checkpoints") if p.is_dir()]
print(f"  .DS_Store files: {len(ds)}")
print(f"  .ipynb_checkpoints dirs: {len(ckpt)}")

print("\nScan complete. Cross-check the counts above against Sections 1-8 below.")


Scanning: /Users/ibby/dev/LJMU/7030DATSCI-Data-Science-Project/7030DATSCI

1. Notebook duplicates (by phase prefix)

2. Report / dissertation draft duplicates
  reports/*.docx -> 4 file(s)
    - reports/07_05_2026_DATSCI7030_Data_Collection_Report.docx  (2279 KB, modified 2026-05-08)
    - reports/08_05_2026_DATSCI7030_Data_Collection_Report.docx  (2547 KB, modified 2026-05-08)
    - reports/DATSCI7030_Data_Collection_Report.docx  (2278 KB, modified 2026-05-07)
    - reports/DATSCI7030_Data_Collection_Report_.docx  (2278 KB, modified 2026-05-07)
  reports/dissertation/*.docx -> 1 file(s)
    - reports/dissertation/7030DATSCI-04_07_2026.docx  (3444 KB, modified 2026-07-04)

3. Missing README.md
  MISSING: notebooks/README.md
  MISSING: src/README.md
  MISSING: scripts/README.md
  MISSING: tests/README.md
  MISSING: reports/README.md
  MISSING: reports/figures/README.md
  MISSING: reports/dissertation/README.md
  MISSING: data/README.md
  MISSING: docs/README.md

4. Potential secrets com

## 1. 🔴 P0 — Exposed Credentials

**Problem**
Live-looking credentials sit on disk in plaintext in two places: `config.yaml` (`api_keys.newsapi`, `api_keys.fred`) and `notebooks/.env` (a Hugging Face token, plus a comment containing what reads as a username and password). A duplicate, now-archived Phase-1 notebook also has a saved cell **output** echoing a partial FRED key (`FRED API key : set (0790aba0...)`).

**Update — 2026-07-04:** `.gitignore` now explicitly covers `config.yaml`, `.env`/`*.env`, and `notebooks/archive/` (where the leaking notebook now lives), and this was verified with `git check-ignore -v` — none of the three will enter git history at the next commit. That closes the *version-control* exposure, but not the underlying problem: **the keys themselves have not been rotated**, and the leaked fragment has not been scrubbed from the archived notebook's saved output. A key sitting safely un-committed on a laptop disk is still a key that could leak via a zip export, a backup, or a future `.gitignore` edit.

**Reason**
A leaked API key is a nuisance (rotate it, done). A leaked Hugging Face token is a nuisance. A password sitting in a comment is not a data-science problem — it's a personal-security problem, and it has no place in a dissertation repository regardless of how it got there.

**Recommended Change**
1. Rotate/revoke the Hugging Face token and both API keys today, independent of any code change.
2. Change the password referenced in `notebooks/.env` if it is reused anywhere else.
3. Remove all hardcoded secrets from `config.yaml`; load them from environment variables instead (`os.environ["FRED_API_KEY"]`), with `config.yaml` only holding the *names* of expected env vars.
4. Keep `notebooks/.env` (or move it to repo root) but treat it as sacred: real values only ever live there locally, never referenced by value in any notebook output, and it must never leave the `.gitignore`d state — **verified 2026-07-04** via `git check-ignore -v notebooks/.env` (matched, excluded).
5. Add a `config.example.yaml` / `.env.example` with placeholder values so the setup instructions in the README are reproducible for an examiner without ever needing the real keys.
6. Delete or scrub the archived duplicate notebook (`notebooks/archive/__01_data_collection__.ipynb`) — it is gitignored so it is not a commit risk, but the partial key in its saved output should not persist on disk indefinitely.

**Files Affected**
`config.yaml`, `notebooks/.env`, `src/data_collector.py` (reads `config.yaml` for keys), `.gitignore`.

**Implementation Steps**
Rotate keys → strip secrets from `config.yaml` → add `.env.example` → update `src/data_collector.py` to read from environment → confirm `.gitignore` covers real `.env` before first commit.

**Dissertation Impact**
Indirect but real: a marker who opens `config.yaml` and finds live credentials will reasonably question the project's engineering rigor, which is one of the explicit assessment criteria (reproducible, version controlled, professional repository).


## 2. 🔴 P0 — No True Market-Only Baseline for RQ3

**Problem**
RQ3 asks whether *event-informed* ML models outperform a *market-only baseline*. `data/processed/model_comparison.parquet` currently compares three models — LASSO, XGBoost, LightGBM — but all three appear to be trained on the same 52-column `model_features.parquet`, which already contains sentiment and event columns (`mean_car`, `n_monetary_events`, `sent_mean_5d`, etc.). There is no model in the comparison table trained on price/technical features only. As it stands, the project cannot yet make the RQ3 claim — there is nothing "market-only" to outperform.

A second thing the live numbers show: XGBoost has a large train/test gap (train R² 0.554, Dir. Acc 0.753 → test R² 0.030, Dir. Acc 0.524), i.e. it is memorising the training window rather than generalising. LASSO is the most stable model (train R² 0.092 → test R² 0.033) and is already flagged as `best_model` in `models/model_metadata.json` — but "most stable of three overfit-prone learners" is a different claim from "beats a market-only baseline," and the dissertation should be precise about which claim it's making.

**Reason**
This is the single research-validity gap most likely to be raised by an examiner, because it sits directly under RQ3 in the marking scheme. Everything else in this report is hygiene; this one is methodology.

**Recommended Change**
1. Add one deliberately minimal baseline model trained on price/technical features only (no sentiment, no event columns, no macro) — even a naive AR(1)/persistence model or a LASSO on the 13 "price & returns" features from the Phase 5 feature groups table would do.
2. Re-run the same test-period evaluation (RMSE, MAE, R², directional accuracy) for that baseline and add it as a row in `model_comparison.parquet`.
3. In Notebook 07/08, explicitly frame the result as "event-augmented model vs. price-only baseline," not "best of three event-augmented models."
4. Report both statistical significance (is the improvement real) and economic significance (is 52-56% directional accuracy tradeable after costs) — current directional accuracy is close to a coin flip and the write-up should not overclaim.
5. Note the XGBoost overfitting explicitly in the evaluation notebook rather than letting the comparison table speak for itself — a marker will notice the gap even if the text doesn't mention it.

**Files Affected**
`06_model_training.ipynb`, `07_model_evaluation.ipynb`, `data/processed/model_comparison.parquet`, `src/features.py`, `src/models.py`.

**Implementation Steps**
Define price-only feature subset → train baseline with identical CV/split protocol → append to comparison table → update Notebook 07/08 narrative and any dissertation chapter that currently states or implies RQ3 is answered.

**Dissertation Impact**
Direct and high — this is the evidentiary basis for one of the three primary research questions.


## 3. 🔴 P0 — No Version Control

**Problem**
**Update — 2026-07-04: partially resolved.** The repository is now a git repository — `git init` has been run (`git status` returns `On branch main`) — but **zero commits have been made**. Every principle in the project brief that depends on version control — reproducibility of history, ability to show iteration, safe experimentation via branches — still has no real support until the first commit lands.

**Reason**
"Version controlled" is one of the seven stated project principles, and a repository with a `.gitignore` but no `.git` reads as unfinished rather than deliberate.

**Recommended Change**
1. Rotate the exposed credentials first (Section 1) — do this before the first commit, not after.
2. ~~`git init`~~ **done.** `.gitignore` coverage re-verified 2026-07-04 with `git check-ignore -v` and a full `git status --porcelain -uall` dry run: `data/raw/`, `data/processed/`, `data/interim/`, `data/external/`, `models/`, `config.yaml`, `.env`, `.DS_Store`, `.ipynb_checkpoints/`, and `notebooks/archive/` all correctly excluded; only source, docs, notebooks, tests, and the dissertation `.docx` would be staged. **The initial commit itself is still not made** — staging list and message are prepared (see founder instructions / README) but intentionally left for the founder to run after confirming Section 1.
3. Decide and document a policy for large binary notebooks (some are 1-2 MB with embedded output) — consider `nbstripout` or committing cleared-output copies, since re-running notebooks with cached data is cheap but diffing megabyte-sized JSON blobs in git is not.
4. Push to a private remote (GitHub/GitLab) so there is an off-machine backup — currently the entire project exists on one disk.

**Files Affected**
Whole repository; `.gitignore`.

**Implementation Steps**
Rotate secrets → `git init` → verify `.gitignore` coverage with `git status` before first `add` → initial commit → add remote → push.

**Dissertation Impact**
Indirect — examiners may ask for the repository or commit history as evidence of process; right now there is none to show.


## 4. 🟠 P1 — Duplicate Files With No Single Source of Truth

**Problem — Notebooks. Resolved 2026-07-04.** Three files claimed to be "Phase 1 — Data Collection". `01_data_collection.ipynb` (37 KB, v2.0, "narrowed scope") is now confirmed and kept as canonical. `__01_data_collection__.ipynb` (942 KB, v1.1, full output, and the source of the partial FRED-key leak in Section 1) has been moved to `notebooks/archive/` (gitignored). The third candidate this section originally referenced, `01_data_collection_narrowed_v2.ipynb`, no longer exists as a live file — it appears to have been renamed directly into the current `01_data_collection.ipynb` (the title cell of the canonical notebook reads "Version 2.0 narrowed"), which is consistent with that history.

**Problem — Reports. Still open, and slightly worse.** `reports/` still contains four near-identical "Data Collection Report" `.docx` files (`DATSCI7030_Data_Collection_Report.docx`, `DATSCI7030_Data_Collection_Report_.docx` — note the trailing underscore — and two dated versions from 07/05 and 08/05/2026). `reports/dissertation/` now has **three** dissertation-related files, not two: `7030DATSCI-04_07_2026.docx`, `7030DATSCI-04_07_2026_reviewed.docx` (both dated today), and `DATSCI7030_Dissertation_Ibrahim_Haroun.docx` (2026-05-31). A Word lock file (`~$30DATSCI-04_07_2026_reviewed.docx`) was also found alongside them — harmless (it just means the reviewed draft was open in Word during the scan) and now covered by a new `~$*` rule added to `.gitignore` 2026-07-04, but it underlines that this folder needs a single-canonical-file pass.

**Reason**
The project brief is explicit: "never create duplicate files." Beyond the stated rule, duplicate notebooks are a direct reproducibility risk — if `02_eda.ipynb` was run against outputs from the "narrowed_v2" collector but `01_data_collection.ipynb` is the one a future run would execute, results silently stop matching. Duplicate dissertation drafts risk citing/submitting the wrong one.

**Recommended Change**
1. Notebooks: confirm which of the three actually produced the current `data/raw/` parquet files (check file timestamps against `data/raw/*.parquet` mtimes, or just re-run the candidate and diff outputs). Keep exactly one as `01_data_collection.ipynb`; delete or move the other two into a `notebooks/archive/` folder (outside the numbered pipeline) with a one-line note on why they were superseded — don't delete history you might want to cite in the dissertation as "an earlier, broader scope was narrowed for X reason."
2. Reports: keep the single most recent `.docx` in `reports/dissertation/` as the canonical draft; move the older one to an `archive/` subfolder or delete once diffed for anything not carried forward. The four "Data Collection Report" files in `reports/` duplicate content that already lives, more accurately, in the README's "Phase Progress" tables — consider whether standalone phase reports are needed at all, or whether the README + dissertation chapter supersede them.
3. Once a single canonical file exists for each, git (Section 3) makes future "which version is current" questions unnecessary — that's exactly the problem version control solves.

**Files Affected**
`notebooks/01_data_collection.ipynb`, `notebooks/__01_data_collection__.ipynb`, `notebooks/01_data_collection_narrowed_v2.ipynb`, `reports/*.docx`, `reports/dissertation/*.docx`.

**Implementation Steps**
Diff the three Phase-1 notebooks → confirm the authoritative one against `data/raw/` timestamps → archive or delete the others → repeat the "keep one, archive the rest" pattern for both report locations.

**Dissertation Impact**
Direct — an examiner (or future you) needs one unambiguous notebook per phase and one unambiguous dissertation file to assess.


## 5. 🟠 P1 — READMEs Describe Files That Don't Exist

**Problem**
`data/raw/README.md` documents `prices.parquet`, `fred_macro.parquet`, and `gdelt_events.parquet` — the actual files on disk are `prices.parquet` (present, matches), but `macro_indicators.parquet` and `gdelt_sample.parquet` (not `fred_macro.parquet` / `gdelt_events.parquet`). `data/processed/README.md` documents `features.parquet` and `target.parquet`, neither of which exists — the actual processed directory has eleven files (`car_results.parquet`, `causal_estimates.parquet`, `daily_sentiment.parquet`, `evaluation_summary.parquet`, `events_tagged.parquet`, `feature_metadata.parquet`, `gdelt_daily_risk.parquet`, `high_impact_events.parquet`, `model_comparison.parquet`, `model_features.parquet`, `scaler.pkl`, `shap_values.parquet`, `test_predictions.parquet`) that the README doesn't mention at all. `data/external/README.md` documents four reference CSVs (`sp500_constituents.csv`, `fomc_dates.csv`, `earnings_calendar.csv`, `macro_events.csv`) — the folder is empty except for the README itself.

**Reason**
These READMEs read as written *before* the pipeline was built and never revisited — which is a normal thing to happen mid-project, but stale data dictionaries are exactly the kind of inconsistency the project brief asks to be caught during review, and they actively mislead anyone (including you, in six months) trying to understand what a folder contains without opening every file.

**Recommended Change**
Regenerate all three `data/*/README.md` files from what is actually on disk. Since this is exactly the kind of fact that goes stale again the moment a notebook is re-run, consider having each collection/processing notebook's last cell auto-write a small `_manifest.json` (filename, shape, generating notebook, last-updated) into the folder, and have the README point at that manifest rather than hand-listing files.

**Files Affected**
`data/raw/README.md`, `data/processed/README.md`, `data/external/README.md`.

**Implementation Steps**
List actual files with `pandas.read_parquet(...).shape` for each → rewrite the three READMEs → optionally add the auto-manifest step to Notebooks 01, 02, 05, 06.

**Dissertation Impact**
Indirect — but a marker who spot-checks a README against `data/` and finds it wrong will spot-check everything else more skeptically.


## 6. 🟠 P1 — Missing README.md in Ten Folders

**Problem — partially resolved 2026-07-04.**
Per the project's own documentation standard ("every important folder should contain a README explaining Purpose, Contents, Input, Output, Dependencies"), the following folders had no README: `notebooks/`, `src/`, `models/`, `scripts/`, `tests/`, `reports/`, `reports/figures/`, `reports/dissertation/`, `data/` (top-level), `docs/`. `models/README.md` has now been added (alongside a new `data/interim/README.md` and `logs/README.md` for two folders created this session). **Still missing:** `notebooks/`, `src/`, `scripts/`, `tests/`, `reports/`, `reports/figures/`, `reports/dissertation/`, `data/` (top-level), `docs/` (top-level — `docs/architecture/` has one, but `docs/` itself doesn't, though `docs/00_project_workflow.md` now partially fills that role).

**Reason**
`src/` in particular has six substantial modules (`data_collector.py`, `event_detector.py`, `causal_engine.py`, `features.py`, `models.py`, `evaluation.py`) with good individual docstrings but no single page explaining how they compose into the pipeline — someone (an examiner, or you after a break) has to open all six files to understand the architecture rather than reading one README.

**Recommended Change**
Add a short README to each folder above. For `src/` specifically, one paragraph per module plus the dependency chain (`data_collector` → `event_detector` → `causal_engine`/`features` → `models` → `evaluation`) is enough — don't over-engineer this into a second architecture document when `docs/architecture/` already exists for diagrams.

**Files Affected**
New `README.md` in each of the ten folders listed above.

**Implementation Steps**
Write folder-by-folder, shortest first (`tests/`, `scripts/` — a few lines each) before the two that need more thought (`src/`, `notebooks/`).

**Dissertation Impact**
Indirect — supports the "well documented" and "easy to understand" principles an examiner will be assessing the repository against, separately from the dissertation text itself.


## 7. 🟡 P1/P2 — Notebook Pipeline Diverges From the Governing 10-Stage Spec

**Problem**
The project brief defines a 10-notebook pipeline (`01_data_collection` … `10_final_results`, with `05_sentiment_analysis` and `09_explainability`/`10_final_results` as distinct stages). The repository instead has 8 notebooks with different names: `01_data_collection`, `02_eda`, `03_event_detection`, `04_causal_analysis`, `05_feature_engineering`, `06_model_training`, `07_model_evaluation`, `08_results_visualisation`. Sentiment analysis lives inside `03_event_detection.ipynb` rather than its own notebook; SHAP/explainability lives inside `07_model_evaluation.ipynb` rather than its own notebook.

**Reason**
This isn't necessarily wrong — folding sentiment scoring into event detection and SHAP into evaluation is a reasonable, less fragmented design, and the brief itself says "avoid unnecessary complexity." The ambiguity itself is now **documented but not yet resolved**: `docs/00_project_workflow.md` (added 2026-07-04) lays out the canonical Phase 0–10 structure and RQ-traceability table, and explicitly flags this exact 8-vs-10 divergence as "an open decision" rather than leaving it implicit. The underlying choice — (a) keep 8 and document it, or (b) split to match 10 — still has not been made.

**Recommended Change**
Make the choice explicit rather than leaving it implicit. Either (a) keep the current 8-notebook structure and add one paragraph to the root README stating that sentiment and explainability were deliberately merged into adjacent phases, and update the project's own instruction set to match reality — or (b) if you'd rather match the original 10-stage spec exactly (useful if the dissertation methodology chapter is structured around 10 phases), split `03_event_detection.ipynb` into `03_event_detection` + `05_sentiment_analysis`, and `07_model_evaluation.ipynb` into `08_model_evaluation` + `09_explainability`. Given Phases 1-5 are already "Complete" per the README, (a) is almost certainly the lower-risk choice at this stage of the project.

**Files Affected**
Root `README.md` (Notebooks table), project instructions/config, optionally `03_event_detection.ipynb`, `07_model_evaluation.ipynb` if you choose to split.

**Implementation Steps**
Decide (a) vs (b) → update README Notebooks table and/or split notebooks accordingly → keep the dissertation methodology chapter's phase numbering consistent with whichever you choose.

**Dissertation Impact**
Direct on presentation, not on results — a methodology chapter that says "Phase 5: Sentiment Analysis" needs to point at where that logic actually lives.


## 8. 🟡 P2 — Optional Datasets: Verify They Earn Their Place

**Problem**
`config.yaml` lists `SPY`, `QQQ`, `GLD`, `TLT` as tickers to collect, and `data/raw/prices.parquet` presumably holds all four (per `data/raw/README.md`). None of `QQQ`, `GLD`, or `TLT` appear anywhere in the README's Phase 3-5 results sections, which only discuss SPY, VIX, and event/sentiment features. Separately, GDELT is wired all the way through to a feature (`gdelt_daily_risk.parquet`, and an `n_...` style feature would presumably use it) despite the README itself stating the GDELT data is only a 5-day proof-of-concept sample, with the full 2015-2025 series explicitly flagged as "the critical data gap."

**Reason**
The brief is explicit: "Only include optional datasets if they improve the analysis" and "Depth is preferred over breadth." Collecting QQQ/GLD/TLT costs little, but if they're not used anywhere downstream they're dead weight in the data dictionary and an easy question for an examiner ("why is GLD here?") that currently has no answer in the documentation. The GDELT gap is more serious: a feature built on a 5-day sample is not a real feature, and if `gdelt_daily_risk.parquet` is currently feeding into `model_features.parquet` on the strength of 5 days of data, that's a data-quality issue baked into the ML layer.

**Recommended Change**
1. Either use QQQ/GLD/TLT in a clearly-scoped comparison (e.g. "does the SPY effect generalise to other asset classes?") or drop them from `config.yaml` and `data/raw/` and note in the README that they were evaluated and excluded — either is defensible, silence is not.
2. Resolve the GDELT gap explicitly before Phase 6/7 finalise: either backfill the full 2015-2025 GDELT series before it's used as a feature, or exclude the GDELT-derived feature from the final model and note why in the feature engineering notebook.

**Files Affected**
`config.yaml`, `data/raw/README.md`, `06_feature_engineering.ipynb` (or wherever `gdelt_daily_risk.parquet` is consumed), root README.

**Implementation Steps**
Grep `src/features.py` and `05_feature_engineering.ipynb` for any column derived from `gdelt_daily_risk` or from QQQ/GLD/TLT → decide keep-and-justify vs. drop-and-document for each → update config and READMEs accordingly.

**Dissertation Impact**
Direct if any GDELT-derived feature is currently in the 52 selected features feeding the final models — a feature built on 5 days of data materially weakens any claim resting on it.


## 9. 🟡 P2 — Housekeeping

Lower-stakes items, worth a single cleanup pass once the P0/P1 items above are decided.

- **Filesystem clutter — attempted 2026-07-04, partially blocked.** `.DS_Store` and `.ipynb_checkpoints/` appear in most folders; confirmed correctly excluded in `.gitignore` (both patterns verified with `git check-ignore`), so there is no commit risk either way. A cleanup pass deleted most of them, but several files under `notebooks/.ipynb_checkpoints/` and a handful of `.DS_Store` files could not be removed due to filesystem permissions in that session — safe to leave (gitignored), finish with a local `find . -name ".DS_Store" -delete` when convenient.
- **`environment.yml` referenced by the suggested structure but absent.** Only `requirements.txt` exists. If you work in conda rather than venv day-to-day, add one; otherwise, simplest fix is removing the `environment.yml` expectation from the suggested structure rather than maintaining two dependency files that can drift apart. `requirements.txt` itself is well-documented and pinned sensibly — no changes needed there.
- **`data/raw/app_csv/` (~120 paginated scrape CSVs)** is undocumented — it's the intermediate output of the APP scraper before consolidation into `app_presidential_documents.parquet`. Worth one line in `data/raw/README.md` explaining it's raw scrape cache rather than a second copy of the same data, so it doesn't look like clutter.
- **`reports/presentation/`** is described in the root README's repository-structure diagram but doesn't exist on disk. Either create it when the slide deck exists, or drop it from the diagram until then.
- **`src/` flat module layout** (six top-level `.py` files) diverges from the suggested `src/{config,data,features,models,visualization,utils}/` subpackage layout in the project brief. Given the project size and the brief's own "avoid unnecessary complexity" principle, six clearly-scoped, well-documented modules is arguably the *right* call, not a shortfall — flagging this as a deliberate decision worth stating in `src/README.md` (Section 6) rather than something to restructure.
- **`models/` flat layout** (no `trained/`/`saved/` split) — same reasoning as above; nine model artefacts flat in one folder is still easy to navigate. No action recommended.

> Founder note:


## 10. Prioritised Action Plan

Tick items off as you close them; re-run Section 0's scan cell afterward to confirm.

### 🔴 P0 — do first
- [ ] Rotate the Hugging Face token and both API keys (newsapi, fred); change the password referenced in `notebooks/.env` if reused elsewhere.
- [ ] Strip hardcoded secrets from `config.yaml`; switch to environment variables; add `.env.example`.
- [x] ~~Decide the authoritative Phase-1 notebook among the three candidates and archive/delete the other two.~~ **Done 2026-07-04** — `01_data_collection.ipynb` (v2.0, narrowed scope) confirmed canonical; `__01_data_collection__.ipynb` (v1.1, contains a partial FRED-key leak in a saved cell output) moved to `notebooks/archive/`, which is gitignored so it can never be committed.

- [ ] **NEW 2026-07-04 (verification pass):** remove the nested git repository at `7030DATSCI/.git` (single "Initial commit" from `Videnify Solutions <info@videnify.com>`, containing only a `LICENSE`) before staging anything — a nested `.git` will either be silently skipped by `git add -A` or recorded as a broken gitlink with no `.gitmodules`, either way it must go first.
- [ ] ~~`git init`~~ **done** — repo initialised, zero commits. ~~verify `.gitignore` coverage~~ **done 2026-07-04**, confirmed via `git check-ignore` + a full untracked-file dry run. **Still open:** remove nested repo (above) → rotate secrets → make the initial commit → push to a private remote.
- [ ] Add a genuine market-only baseline model to `model_comparison.parquet` and re-frame the RQ3 narrative around it.
- [ ] Investigate and document the XGBoost train/test overfitting gap in the evaluation notebook.

### 🟠 P1 — before submission
- [ ] Consolidate the four "Data Collection Report" `.docx` files in `reports/` into one (or fold into README/dissertation).
- [x] ~~Consolidate the dissertation-related files in `reports/dissertation/` into one canonical file.~~ **Resolved by 2026-07-04 (verification pass)** — folder now holds a single file, `7030DATSCI-04_07_2026.docx`; the two extra drafts noted earlier in §4 are gone.
- [ ] Regenerate `data/raw/README.md`, `data/processed/README.md`, `data/external/README.md` from what's actually on disk.
- [ ] Add missing `README.md` to: `notebooks/`, `src/`, `scripts/`, `tests/`, `reports/`, `reports/figures/`, `reports/dissertation/`, `data/` (top-level), `docs/` (top-level). ~~`models/`~~ **done 2026-07-04** (plus new `data/interim/README.md`, `logs/README.md` for two folders created this session).
- [ ] **NEW 2026-07-04 (verification pass):** `src/*.py` docstring headers have `Description` and `Usage` but are missing `Author`, `Version`, and `Dependencies`, which the project's own script-documentation standard requires — true of all six modules (`data_collector.py`, `event_detector.py`, `causal_engine.py`, `features.py`, `models.py`, `evaluation.py`).
- [ ] Decide whether the 8-notebook pipeline is a deliberate simplification of the 10-stage brief, or split notebooks to match it exactly. **Partially done 2026-07-04:** the divergence is now explicitly documented in `docs/00_project_workflow.md`; the actual (a)-vs-(b) decision is still open.
- [ ] Extend the root README's Phase Progress section to cover Phases 6-8 in the same depth as Phases 3-5.

### 🟡 P2 — polish
- [ ] Delete `.DS_Store` files and `.ipynb_checkpoints/` folders throughout. **Attempted 2026-07-04** — most removed; a few blocked by sandbox filesystem permissions, harmless since both are gitignored.
- [ ] Resolve QQQ/GLD/TLT: use-and-justify or drop-and-document.
- [ ] Resolve the GDELT 5-day-sample gap before any GDELT-derived feature is treated as final.
- [ ] Document `data/raw/app_csv/` as scrape cache in `data/raw/README.md`.
- [ ] Reconcile `reports/presentation/` — create it or remove it from the README structure diagram.
- [ ] Note the flat `src/` and `models/` layouts as a deliberate, size-appropriate choice in their new READMEs.

> Founder note:


## 10a. Session Update — 2026-07-04 Cleanup Pass

Founder-requested repository cleanup and workflow update, tracked against the founder's own P0/P1 grouping (distinct from — and complementary to — the detailed action plan in Section 10 above).

### P0 — this session
- ✅ **Security cleanup (partial)** — full repo secret scan run; findings documented in §1 above. Exposed values are in gitignored files (`config.yaml`, `notebooks/.env`) so nothing will be committed, but **the keys themselves have not been rotated** and the leaked partial FRED key inside the archived duplicate notebook's output has not been scrubbed. Rotation is a founder action this assistant cannot perform — treat as still open until done.
- ✅ **Canonical Notebook 01 selected** — `notebooks/01_data_collection.ipynb`. Duplicate archived to `notebooks/archive/` (gitignored).
- ✅ **`.gitignore` updated** — rewritten to ignore directory *contents* (`data/raw/*`, `models/*`, `logs/*`, etc.) rather than whole directories, with `README.md`/`.gitkeep` explicitly negated back in so folder structure survives a fresh clone. Added `data/interim/`, `*.xlsx`, `*.key`, `*.pem`, broadened `.env` coverage, and `notebooks/archive/`.
- ⏳ **Git initial commit prepared, not run** — staging list and commit message are ready (see §3 / founder instructions); commit is intentionally left for the founder to execute after confirming the P0 security items above.

### P1 — flagged for a future session (not actioned now)
- ⬜ **Research Bible documentation** — not started.
- ⬜ **Statistics plan** — not started (event-study test choices, multiple-comparison handling, etc.).
- ⬜ **Baseline model plan** — not started; ties directly into the existing §2 finding ("No True Market-Only Baseline for RQ3").
- ⬜ **Dissertation alignment** — not started; `docs/00_project_workflow.md` now documents the official Phase 0–10 pipeline and RQ traceability table this should be checked against.

### Also done this session (repository hygiene, not in either P-list above)
- `README.md` — added Research Questions, Hypotheses, Data Policy, expanded Reproducibility, and a Current Project Status section.
- `docs/00_project_workflow.md` — created; documents Phases 0–10 and the RQ-traceability rule.
- `data/interim/`, `models/`, `logs/` — given README.md + .gitkeep so they survive as empty, documented folders under the new content-only ignore rules.
- `.DS_Store` / `.ipynb_checkpoints/` cleanup (§9 P2 item) — attempted; most removed, a handful of files under `notebooks/.ipynb_checkpoints/` and scattered `.DS_Store` files could not be deleted due to sandbox filesystem permissions in this session. No risk either way — both patterns are gitignored — but a manual `find . -name '.DS_Store' -delete` / removing `**/.ipynb_checkpoints` locally would finish the job.


## 10b. Session Update — 2026-07-04 Verification Pass

Independent re-scan requested by the founder ("full repository review"). Purpose: confirm nothing in Sections 1-9 has silently regressed or resolved since 10a, and surface anything the original scan didn't check for. Nothing was deleted, renamed, or overwritten to produce this update, consistent with the project rule to analyse before acting.

### Confirmed unchanged (still open)
- Credentials in `config.yaml` (`api_keys.newsapi`, `api_keys.fred`) and `notebooks/.env` — **not rotated**. §1 stands as written.
- Git — repository still has **zero commits**. §3 stands as written.
- `reports/*.docx` — still four near-duplicate "Data Collection Report" files. §4 (reports half) stands as written.
- `data/raw/README.md` — still names `fred_macro.parquet` / `gdelt_events.parquet`; actual files are `macro_indicators.parquet` / `gdelt_sample.parquet`. §5 stands as written.
- Nine folders (`notebooks/`, `src/`, `scripts/`, `tests/`, `reports/`, `reports/figures/`, `reports/dissertation/`, `data/`, `docs/`) still have no `README.md`. §6 stands as written.
- GDELT — still referenced in `src/data_collector.py` and consumed in `02_eda.ipynb`, `03_event_detection.ipynb`, and `08_results_visualisation.ipynb`; the 5-day-sample depth question in §8 has not been independently re-verified this pass and should still be treated as open.

### Corrected (resolved since 10a — better than last recorded)
- **§4, reports/dissertation/:** previously reported as three files (plus a Word lock file). Now confirmed as **one** — `7030DATSCI-04_07_2026.docx` (3.4 MB, 2026-07-04). The `_reviewed.docx` and the 2026-05-31 draft are no longer present. Marked resolved in the Action Plan (§10).

### New — 🔴 P0 — Nested Git Repository

**Problem**
`7030DATSCI/` (a subdirectory *inside* the project root, same name as the project itself) contains its own `.git`, with one commit — `"Initial commit"`, authored by `Videnify Solutions <info@videnify.com>` at 2026-07-04 21:59 — adding a single `LICENSE` file. This looks like a fresh GitHub repository (created with an auto-generated license) that was cloned into the project folder instead of somewhere else, rather than anything intentional in this project's design.

**Reason**
A `.git` directory nested inside another git working tree is not a subfolder as far as the parent repo is concerned — it is either silently invisible to `git status`/`git add -A` (parent git sees an empty-looking directory and skips it) or, if force-added, becomes a **gitlink** (a submodule reference) with no corresponding `.gitmodules` entry, which produces a broken reference the moment someone clones the parent repository. Either outcome is worse than the current not-yet-committed state, and this needs to be resolved *before* the initial commit that §3/§10 already has queued — fixing it after the fact means rewriting history.

**Recommended Change**
1. Confirm the nested `LICENSE` file isn't needed (it reads as GitHub's default MIT/Apache template text, not project-specific).
2. If it is not needed: delete `7030DATSCI/.git` and `7030DATSCI/LICENSE` (or the whole `7030DATSCI/` subfolder) entirely.
3. If a LICENSE file for the project itself is wanted: copy just the license text to the project root as `LICENSE` (no nested `.git`), then remove the nested repo.
4. Re-run a `find . -name .git -type d` sweep immediately before the first `git add` to confirm no other nested repos exist.

**Files Affected**
`7030DATSCI/` (nested directory — its `.git/` and `LICENSE`).

**Implementation Steps**
Decide keep-license-text vs. discard → copy text to root `LICENSE` if keeping → delete the nested `7030DATSCI/` subfolder and its `.git` → confirm with `git status` that the parent repo's working tree looks as expected → proceed with the initial commit already staged per §3.

**Dissertation Impact**
Indirect but compounding with §3 — an examiner who clones the repository and hits a broken submodule reference will read it as an engineering-process failure independent of the research content.

### New — 🟠 P1 — Script Headers Incomplete

**Problem**
The project's own documentation standard requires every script to include `Description`, `Author`, `Version`, `Usage`, and `Dependencies`. All six `src/*.py` modules (`data_collector.py`, `event_detector.py`, `causal_engine.py`, `features.py`, `models.py`, `evaluation.py`) have a clear one-paragraph `Description` and a runnable `Usage` example, but none currently states an `Author`, a `Version`, or its `Dependencies` explicitly in the header docstring — those facts exist only implicitly (imports at the top of the file, and authorship inferred from git history once §3 gives the repo one).

**Reason**
This is a documentation-completeness gap against a standard the project set for itself, not a functional problem — the code runs regardless. But it's a fast, mechanical fix, and it directly supports the "well documented" and "reproducible" principles: a reader should be able to tell a module's dependency footprint (e.g. `event_detector.py` needs `transformers`/`torch`/`spacy`) from its own header without cross-referencing `requirements.txt`.

**Recommended Change**
Add three lines to each of the six module docstrings: `Author: Ibrahim Haroun`, `Version:` (align with `config.yaml`'s `project.version: 1.1.0` or track per-module if they've diverged independently), and `Dependencies:` (the 2-4 packages each module actually imports beyond the standard library, e.g. `pandas, numpy` for `features.py`; `transformers, torch, spacy` for `event_detector.py`).

**Files Affected**
`src/data_collector.py`, `src/event_detector.py`, `src/causal_engine.py`, `src/features.py`, `src/models.py`, `src/evaluation.py`.

**Implementation Steps**
Pick one canonical header template → apply to all six files → cross-check the `Dependencies` line against each file's actual `import` statements (not `requirements.txt`, which is broader than any single module needs) → mention this convention once in the new `src/README.md` (already queued in §6/§10) so future modules follow the same header.

**Dissertation Impact**
Indirect — polish, not evidence; safe to batch with the broader README pass in §6.

> Founder note:


## 11. Founder Notes (freeform)

Use this space for anything that doesn't fit neatly above — open questions for your supervisor, scope decisions, or things you want revisited next review pass.

> Founder note:
